
# 260523 · 空间模态编码 MVP 训练（Colab 优化版）

对齐开题报告：**异构 GNN 拓扑编码 + VAE 潜空间 + 约束条件注入 + 3D 体素解码**。

| 模块 | 原版 260510 | 本版优化 |
|---|---|---|
| 数据集 | Drive 旧路径 ~96 样本 | 升级 JSON（采光字段 + metadata.stats） |
| 节点特征 | 5 维几何 | 8 维（几何 + 采光 access/priority/ratio） |
| 潜空间 | 无 VAE | Graph VAE（mu/logvar + KL） |
| 约束 | 无 | 全局条件向量注入解码器（CVAE MVP） |
| 训练 | 全量 200 epoch | train/val 划分 + 早停 + Dice 稀疏损失 |

**数据准备：** 将仓库 `data/` 上传至 `MyDrive/master_thesis/data/`，或修改下方 `DATA_DIR`。


In [ ]:

# Step 0: 安装依赖（Colab 首次运行）
!pip -q install torch_geometric


In [ ]:

# Step 1: 全局配置 + 数据工具
import os, json, random, glob
import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim
from torch_geometric.data import HeteroData
from torch_geometric.loader import DataLoader as GraphDataLoader
from torch_geometric.nn import HeteroConv, SAGEConv, GATv2Conv, global_mean_pool

USE_DRIVE = True
if USE_DRIVE:
    try:
        from google.colab import drive
        drive.mount('/content/drive')
    except ImportError:
        USE_DRIVE = False

def resolve_data_dir():
    candidates = [
        '/content/drive/MyDrive/master_thesis/data',
        '/content/drive/MyDrive/260509_dataset',
        '/content/data',
        os.path.join(os.getcwd(), 'data'),
        os.path.abspath(os.path.join(os.getcwd(), '..', 'data')),
    ]
    for path in candidates:
        if os.path.isdir(path):
            return path
    return candidates[0]

DATA_DIR = resolve_data_dir()

OUT_DIR = os.path.join(DATA_DIR, 'processed_tensors_v2')
CACHE_VERSION = 'mvp_v2_lighting_cvae'

VOXEL_SIZE = 300.0
RES_X, RES_Y, RES_Z = 64, 128, 32

CHANNEL_MAP = {
    'empty': 0, 'entryway': 1, 'living_room': 2, 'dining_room': 3,
    'kitchen': 4, 'bedroom': 5, 'bathroom': 6, 'corridor': 7,
    'stairs': 8, 'utility': 9, 'balcony': 10, 'multi_purpose': 11,
}
ROOM_TYPES = [k for k in CHANNEL_MAP if k != 'empty']
NUM_CHANNELS = len(CHANNEL_MAP)
NODE_IN_DIM = 8
COND_DIM = 3 + len(ROOM_TYPES) + 3

LIGHTING_ACCESS_MAP = {'direct': 1.0, 'indirect': 0.5, 'none': 0.0}

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'设备: {device}')
print(f'数据目录: {DATA_DIR}  存在: {os.path.isdir(DATA_DIR)}')
print(f'节点维={NODE_IN_DIM}  条件维={COND_DIM}')


def set_seed(seed=42):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)


def get_node_supertype(room_type):
    if room_type in ['bedroom', 'living_room', 'dining_room', 'multi_purpose']:
        return 'living'
    if room_type in ['kitchen', 'bathroom', 'utility', 'balcony']:
        return 'service'
    return 'circulation'


def check_hetero_adjacency(r1, r2, tol=150, min_shared_len=300):
    if r1.get('floor', 1) != r2.get('floor', 1):
        if r1['type'] == 'stairs' or r2['type'] == 'stairs':
            ox = max(0, min(r1['box_max'][0], r2['box_max'][0]) - max(r1['box_min'][0], r2['box_min'][0]))
            oy = max(0, min(r1['box_max'][1], r2['box_max'][1]) - max(r1['box_min'][1], r2['box_min'][1]))
            if ox > 0 and oy > 0:
                return 'vertical'
        return None
    if r1.get('floor', 1) == r2.get('floor', 1):
        ox = max(0, min(r1['box_max'][0], r2['box_max'][0]) - max(r1['box_min'][0], r2['box_min'][0]) + tol)
        oy = max(0, min(r1['box_max'][1], r2['box_max'][1]) - max(r1['box_min'][1], r2['box_min'][1]) + tol)
        if (ox > min_shared_len and oy > 0) or (oy > min_shared_len and ox > 0):
            return 'horizontal'
    return None


def effective_lighting_ratio(room):
    dx = room['box_max'][0] - room['box_min'][0]
    dy = room['box_max'][1] - room['box_min'][1]
    floor_area = max(dx * dy, 1.0)
    eff = room.get('effective_lighting', [])
    total = sum(float(e.get('area', 0)) * float(e.get('attenuation', 1.0)) for e in eff)
    return min(total / floor_area, 5.0) / 5.0


def build_room_features(room, build_min, b_size):
    dx = room['box_max'][0] - room['box_min'][0]
    dy = room['box_max'][1] - room['box_min'][1]
    area = (dx * dy) / 1e6
    aspect = max(dx, dy) / max(min(dx, dy), 1.0)
    cx = ((room['box_min'][0] + room['box_max'][0]) / 2 - build_min[0]) / (b_size[0] + 1e-5)
    cy = ((room['box_min'][1] + room['box_max'][1]) / 2 - build_min[1]) / (b_size[1] + 1e-5)
    cz = ((room['box_min'][2] + room['box_max'][2]) / 2 - build_min[2]) / (b_size[2] + 1e-5)
    lighting_access = LIGHTING_ACCESS_MAP.get(room.get('lighting_access', 'none'), 0.0)
    lighting_priority = float(room.get('lighting_priority', 0)) / 10.0
    lighting_ratio = effective_lighting_ratio(room)
    return [area, aspect, cx, cy, cz, lighting_access, lighting_priority, lighting_ratio]


def build_condition_vector(data):
    meta = data.get('metadata', {})
    stats = meta.get('stats', {})
    bsize = meta.get('building_size', {'x': 1.0, 'y': 1.0, 'z': 1.0})
    rooms = data.get('rooms', [])
    total = max(len(rooms), 1)

    cond = [
        float(bsize.get('x', 1.0)) / 30000.0,
        float(bsize.get('y', 1.0)) / 30000.0,
        float(bsize.get('z', 1.0)) / 9000.0,
    ]
    for rt in ROOM_TYPES:
        cond.append(float(stats.get(rt, 0)) / total)

    direct = indirect = none = 0
    for r in rooms:
        acc = r.get('lighting_access', 'none')
        if acc == 'direct':
            direct += 1
        elif acc == 'indirect':
            indirect += 1
        else:
            none += 1
    cond.extend([direct / total, indirect / total, none / total])
    return cond


def json_to_sample(data):
    rooms = data.get('rooms', [])
    if not rooms:
        return None

    all_coords = np.array([r['box_min'] for r in rooms] + [r['box_max'] for r in rooms])
    build_min = all_coords.min(axis=0)
    build_max = all_coords.max(axis=0)
    b_size = build_max - build_min

    hetero = HeteroData()
    nodes_dict = {'living': [], 'service': [], 'circulation': []}
    id_to_idx = {}

    for r in rooms:
        ntype = get_node_supertype(r['type'])
        id_to_idx[r['id']] = (ntype, len(nodes_dict[ntype]))
        nodes_dict[ntype].append(build_room_features(r, build_min, b_size))

    for ntype, feats in nodes_dict.items():
        hetero[ntype].x = torch.tensor(feats, dtype=torch.float32) if feats else torch.empty((0, NODE_IN_DIM))

    edges_dict = {}
    for i in range(len(rooms)):
        for j in range(i + 1, len(rooms)):
            r1, r2 = rooms[i], rooms[j]
            rel = check_hetero_adjacency(r1, r2)
            if not rel:
                continue
            t1, idx1 = id_to_idx[r1['id']]
            t2, idx2 = id_to_idx[r2['id']]
            for src_t, dst_t, si, di in ((t1, t2, idx1, idx2), (t2, t1, idx2, idx1)):
                key = (src_t, rel, dst_t)
                edges_dict.setdefault(key, []).append([si, di])

    for key, elist in edges_dict.items():
        hetero[key].edge_index = torch.tensor(elist, dtype=torch.long).t().contiguous()

    grid = np.zeros((RES_X, RES_Y, RES_Z), dtype=np.int8)
    phys_center_xy = (build_min[:2] + build_max[:2]) / 2.0
    offset_xy = np.array([RES_X * VOXEL_SIZE / 2, RES_Y * VOXEL_SIZE / 2]) - phys_center_xy
    z_min_phys = build_min[2]

    for r in rooms:
        ch = CHANNEL_MAP.get(r['type'], 0)
        ix_min = int((r['box_min'][0] + offset_xy[0]) / VOXEL_SIZE)
        ix_max = int((r['box_max'][0] + offset_xy[0]) / VOXEL_SIZE)
        iy_min = int((r['box_min'][1] + offset_xy[1]) / VOXEL_SIZE)
        iy_max = int((r['box_max'][1] + offset_xy[1]) / VOXEL_SIZE)
        iz_min = int((r['box_min'][2] - z_min_phys) / VOXEL_SIZE)
        iz_max = int((r['box_max'][2] - z_min_phys) / VOXEL_SIZE)
        grid[max(0, ix_min):min(RES_X, ix_max), max(0, iy_min):min(RES_Y, iy_max), max(0, iz_min):min(RES_Z, iz_max)] = ch

    one_hot = np.zeros((NUM_CHANNELS, RES_X, RES_Y, RES_Z), dtype=np.float32)
    for c in range(NUM_CHANNELS):
        one_hot[c] = (grid == c).astype(np.float32)

    cond = torch.tensor(build_condition_vector(data), dtype=torch.float32)
    return {'graph': hetero, 'voxel': torch.tensor(one_hot, dtype=torch.float32), 'condition': cond}


def dice_loss_with_logits(logits, targets, eps=1e-6):
    probs = torch.sigmoid(logits)
    dims = (0, 2, 3, 4)
    inter = (probs * targets).sum(dims)
    union = probs.sum(dims) + targets.sum(dims)
    dice = (2 * inter + eps) / (union + eps)
    return 1.0 - dice.mean()


def graph_batch_size(batch):
    return int(getattr(batch, 'num_graphs', 1))


def graph_batch_dict(batch):
    if hasattr(batch, 'batch_dict'):
        return batch.batch_dict
    bd = {}
    for ntype in batch.node_types:
        if batch[ntype].x.size(0) > 0:
            if hasattr(batch[ntype], 'batch'):
                bd[ntype] = batch[ntype].batch
            else:
                bd[ntype] = torch.zeros(
                    batch[ntype].x.size(0), dtype=torch.long, device=batch[ntype].x.device
                )
    return bd


def graph_condition(batch):
    bs = graph_batch_size(batch)
    cond = batch.condition
    if cond.dim() == 1:
        cond = cond.unsqueeze(0)
    return cond.view(bs, -1)


def prepare_graph_batch(hg):
    return next(iter(GraphDataLoader([hg], batch_size=1)))


In [ ]:

# Step 2: 批量预处理 JSON -> .pt（含缓存版本号）
os.makedirs(OUT_DIR, exist_ok=True)
meta_path = os.path.join(OUT_DIR, '_cache_meta.json')

json_files = sorted([
    f for f in glob.glob(os.path.join(DATA_DIR, 'house_*.json'))
    if not f.endswith('_topology.json')
])
if not json_files:
    raise FileNotFoundError(f'未找到 JSON: {DATA_DIR}')

need_rebuild = True
if os.path.exists(meta_path):
    with open(meta_path, 'r', encoding='utf-8') as f:
        meta = json.load(f)
    need_rebuild = meta.get('version') != CACHE_VERSION or meta.get('count') != len(json_files)

if need_rebuild:
    ok, fail = 0, 0
    for fp in json_files:
        fname = os.path.basename(fp)
        try:
            with open(fp, 'r', encoding='utf-8') as f:
                data = json.load(f)
            sample = json_to_sample(data)
            if sample is None:
                continue
            torch.save(sample, os.path.join(OUT_DIR, fname.replace('.json', '.pt')))
            ok += 1
        except Exception as exc:
            fail += 1
            print(f'[FAIL] {fname}: {exc}')
    with open(meta_path, 'w', encoding='utf-8') as f:
        json.dump({'version': CACHE_VERSION, 'count': len(json_files), 'ok': ok, 'fail': fail}, f, indent=2)
    print(f'预处理完成: 成功 {ok}, 失败 {fail}, 输出 {OUT_DIR}')
else:
    print(f'命中缓存 {CACHE_VERSION}，跳过预处理（共 {len(json_files)} 个 JSON）')


In [ ]:

# Step 3: CVAE 模型（GNN 编码 + 条件注入 3D 解码）
LATENT_DIM = 256
HIDDEN_DIM = 128


def build_hetero_convs(in_dim, out_dim, vertical_gat=False):
    h_conv = {}
    node_types = ['living', 'service', 'circulation']
    for s in node_types:
        for d in node_types:
            h_conv[(s, 'horizontal', d)] = SAGEConv(in_dim, out_dim)
    vertical_pairs = [
        ('circulation', 'living'), ('living', 'circulation'),
        ('circulation', 'service'), ('service', 'circulation'),
        ('circulation', 'circulation'),
    ]
    for s, d in vertical_pairs:
        if vertical_gat:
            h_conv[(s, 'vertical', d)] = GATv2Conv(
                in_dim, out_dim, heads=2, concat=False, add_self_loops=False
            )
        else:
            h_conv[(s, 'vertical', d)] = SAGEConv(in_dim, out_dim)
    return HeteroConv(h_conv, aggr='sum')


class HeteroGraphVAEEncoder(nn.Module):
    def __init__(self, node_in_dim=NODE_IN_DIM, hidden_dim=HIDDEN_DIM, latent_dim=LATENT_DIM):
        super().__init__()
        self.conv1 = build_hetero_convs(node_in_dim, hidden_dim, vertical_gat=True)
        self.conv2 = build_hetero_convs(hidden_dim, hidden_dim, vertical_gat=False)
        self.to_mu = nn.Linear(hidden_dim, latent_dim)
        self.to_logvar = nn.Linear(hidden_dim, latent_dim)

    def _pool(self, x_dict, batch_dict):
        feats = []
        for ntype, x in x_dict.items():
            if ntype in batch_dict and x.size(0) > 0:
                feats.append(global_mean_pool(x, batch_dict[ntype]))
        if not feats:
            dev = x_dict[list(x_dict.keys())[0]].device
            return torch.zeros(1, HIDDEN_DIM, device=dev)
        return torch.stack(feats, dim=0).mean(dim=0)

    def forward(self, x_dict, edge_index_dict, batch_dict):
        x_dict = {k: torch.relu(self.conv1(x_dict, edge_index_dict)[k]) for k in x_dict}
        x_dict = {k: torch.relu(self.conv2(x_dict, edge_index_dict)[k]) for k in x_dict}
        h = self._pool(x_dict, batch_dict)
        return self.to_mu(h), self.to_logvar(h)


class ConditionalVoxelDecoder(nn.Module):
    def __init__(self, latent_dim=LATENT_DIM, cond_dim=COND_DIM, channels=NUM_CHANNELS):
        super().__init__()
        self.init_volume_size = (256, 4, 8, 2)
        in_dim = latent_dim + cond_dim
        self.fc = nn.Linear(in_dim, int(np.prod(self.init_volume_size)))
        self.decoder = nn.Sequential(
            nn.ConvTranspose3d(256, 128, 4, stride=2, padding=1), nn.BatchNorm3d(128), nn.ReLU(True),
            nn.ConvTranspose3d(128, 64, 4, stride=2, padding=1), nn.BatchNorm3d(64), nn.ReLU(True),
            nn.ConvTranspose3d(64, 32, 4, stride=2, padding=1), nn.BatchNorm3d(32), nn.ReLU(True),
            nn.ConvTranspose3d(32, channels, 4, stride=2, padding=1),
        )

    def forward(self, z, cond):
        x = torch.cat([z, cond], dim=-1)
        d = self.fc(x).view(z.size(0), *self.init_volume_size)
        return self.decoder(d)


class SpatialModalCVAE(nn.Module):
    def __init__(self):
        super().__init__()
        self.encoder = HeteroGraphVAEEncoder()
        self.decoder = ConditionalVoxelDecoder()

    def reparameterize(self, mu, logvar):
        if self.training:
            std = torch.exp(0.5 * logvar)
            eps = torch.randn_like(std)
            return mu + eps * std
        return mu

    def forward(self, batch):
        mu, logvar = self.encoder(
            batch.x_dict, batch.edge_index_dict, graph_batch_dict(batch)
        )
        z = self.reparameterize(mu, logvar)
        cond = graph_condition(batch)
        logits = self.decoder(z, cond)
        return logits, mu, logvar


model = SpatialModalCVAE().to(device)
print(f'参数量: {sum(p.numel() for p in model.parameters()):,}')


In [ ]:

# Step 4: 训练（train/val + 早停 + BCE + Dice + KL）
set_seed(42)

pt_files = sorted(glob.glob(os.path.join(OUT_DIR, '*.pt')))
pt_files = [p for p in pt_files if not os.path.basename(p).startswith('_')]
if not pt_files:
    raise RuntimeError('请先运行 Step 2 预处理')

dataset_list = []
for fp in pt_files:
    try:
        sample = torch.load(fp, weights_only=False)
        hg = sample['graph']
        hg.y = sample['voxel']
        hg.condition = sample['condition'].unsqueeze(0)
        dataset_list.append(hg)
    except Exception as exc:
        print(f'读取失败 {fp}: {exc}')

random.shuffle(dataset_list)
split = max(1, int(len(dataset_list) * 0.8))
train_set, val_set = dataset_list[:split], dataset_list[split:]
print(f'样本总数 {len(dataset_list)} | 训练 {len(train_set)} | 验证 {len(val_set)}')

BATCH_SIZE = 2
train_loader = GraphDataLoader(train_set, batch_size=BATCH_SIZE, shuffle=True)
val_loader = GraphDataLoader(val_set, batch_size=BATCH_SIZE, shuffle=False) if val_set else None

optimizer = optim.Adam(model.parameters(), lr=1e-3, weight_decay=1e-5)
scheduler = optim.lr_scheduler.ReduceLROnPlateau(optimizer, mode='min', factor=0.5, patience=8)

EPOCHS = 120
PATIENCE = 20
KL_WEIGHT = 1e-4
DICE_WEIGHT = 0.3

best_val = float('inf')
bad_epochs = 0
history = {'train': [], 'val': []}
save_path = os.path.join(DATA_DIR, 'spatial_modal_cvae_mvp.pth')

def run_epoch(loader, train_mode=True):
    model.train(train_mode)
    total = 0.0
    n = 0
    for batch in loader:
        batch = batch.to(device)
        bs = graph_batch_size(batch)
        target = batch.y.view(bs, NUM_CHANNELS, RES_X, RES_Y, RES_Z)

        if train_mode:
            optimizer.zero_grad()

        logits, mu, logvar = model(batch)
        bce = F.binary_cross_entropy_with_logits(logits, target)
        dice = dice_loss_with_logits(logits, target)
        kl = -0.5 * torch.mean(1 + logvar - mu.pow(2) - logvar.exp())
        loss = bce + DICE_WEIGHT * dice + KL_WEIGHT * kl

        if train_mode:
            loss.backward()
            torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
            optimizer.step()

        total += loss.item() * bs
        n += bs
    return total / max(n, 1)

print(f'开始训练 ({device}) ...')
for epoch in range(1, EPOCHS + 1):
    tr = run_epoch(train_loader, True)
    history['train'].append(tr)
    va = run_epoch(val_loader, False) if val_loader else tr
    history['val'].append(va)
    scheduler.step(va)

    if va < best_val:
        best_val = va
        bad_epochs = 0
        torch.save(model.state_dict(), save_path)
    else:
        bad_epochs += 1

    if epoch == 1 or epoch % 10 == 0:
        print(f'Epoch {epoch:03d}/{EPOCHS} | train={tr:.4f} val={va:.4f} best={best_val:.4f}')

    if bad_epochs >= PATIENCE:
        print(f'早停于 epoch {epoch}，最佳 val={best_val:.4f}')
        break

print(f'最佳权重已保存: {save_path}')

import matplotlib.pyplot as plt
plt.figure(figsize=(8, 4))
plt.plot(history['train'], label='train')
plt.plot(history['val'], label='val')
plt.title('Spatial Modal CVAE Loss')
plt.xlabel('Epoch')
plt.ylabel('Loss')
plt.grid(True, alpha=0.3)
plt.legend()
plt.show()


In [ ]:

# Step 5: 验证推理 — 对比 GT 与模型输出体素
import plotly.graph_objects as go
import plotly.io as pio
pio.renderers.default = 'colab'

COLOR_DICT = {
    1: '#808080', 2: '#FF8000', 3: '#FFFF00', 4: '#00FF00', 5: '#0000FF',
    6: '#FF0000', 7: '#B0B0FF', 8: '#A000FF', 9: '#3CB371', 10: '#00FFFF', 11: '#FFC0CB',
}
NAME_MAP = {
    1: '玄关', 2: '客厅', 3: '餐厅', 4: '厨房', 5: '卧室', 6: '卫生间',
    7: '过道', 8: '楼梯', 9: '储藏', 10: '阳台', 11: '多功能',
}

weight_path = os.path.join(DATA_DIR, 'spatial_modal_cvae_mvp.pth')
model.load_state_dict(torch.load(weight_path, map_location=device, weights_only=True))
model.eval()

sample_pt = random.choice(pt_files)
sample = torch.load(sample_pt, weights_only=False)
hg = sample['graph']
hg.y = sample['voxel']
hg.condition = sample['condition'].unsqueeze(0)
batch = prepare_graph_batch(hg).to(device)

with torch.no_grad():
    logits, _, _ = model(batch)
    pred = torch.argmax(logits[0], dim=0).cpu().numpy()
    gt = torch.argmax(sample['voxel'], dim=0).numpy()

def voxel_coords(class_map, max_pts=8000):
    coords = np.argwhere(class_map > 0)
    if len(coords) > max_pts:
        coords = coords[np.random.choice(len(coords), max_pts, replace=False)]
    xs, ys, zs, cols, texts = [], [], [], [], []
    for ix, iy, iz in coords:
        cid = int(class_map[ix, iy, iz])
        xs.append(ix * VOXEL_SIZE)
        ys.append(iy * VOXEL_SIZE)
        zs.append(iz * VOXEL_SIZE)
        cols.append(COLOR_DICT.get(cid, '#CCC'))
        texts.append(NAME_MAP.get(cid, str(cid)))
    return xs, ys, zs, cols, texts

from plotly.subplots import make_subplots
fig = make_subplots(rows=1, cols=2, specs=[[{'type': 'scene'}, {'type': 'scene'}]],
                    subplot_titles=('Ground Truth', 'CVAE Prediction'))

for col, arr in ((1, gt), (2, pred)):
    xs, ys, zs, cols, texts = voxel_coords(arr)
    fig.add_trace(go.Scatter3d(
        x=xs, y=ys, z=zs, mode='markers',
        marker=dict(size=3, color=cols, symbol='square', opacity=0.85),
        text=texts, hoverinfo='text', showlegend=False,
    ), row=1, col=col)

acc = (pred == gt).mean()
occupied_acc = (pred[gt > 0] == gt[gt > 0]).mean() if (gt > 0).any() else 0.0
print(f'样本: {os.path.basename(sample_pt)}')
print(f'全网格准确率: {acc*100:.2f}% | 非空体素准确率: {occupied_acc*100:.2f}%')

fig.update_layout(height=650, width=1200, title='空间模态 CVAE 体素重建对比')
fig.update_scenes(aspectmode='data')
fig.show()


## Step 6: 生成效果系统评估

**请按顺序运行：** Step 6a（定义函数）→ Step 6b（批量评估）


In [ ]:
# Step 6a: 评估工具函数（必须先运行本 cell）
import os, json
from collections import defaultdict
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

EVAL_DIR = os.path.join(DATA_DIR, 'eval_reports')
os.makedirs(EVAL_DIR, exist_ok=True)

TYPE_NAMES = {v: k for k, v in CHANNEL_MAP.items() if v > 0}
CN_NAMES = {
    'entryway': '玄关', 'living_room': '客厅', 'dining_room': '餐厅', 'kitchen': '厨房',
    'bedroom': '卧室', 'bathroom': '卫生间', 'corridor': '过道', 'stairs': '楼梯',
    'utility': '储藏', 'balcony': '阳台', 'multi_purpose': '多功能',
}

def logits_to_class(logits):
    return torch.argmax(logits, dim=1 if logits.dim() == 5 else 0).cpu().numpy()

def count_components(mask):
    if not mask.any():
        return 0
    visited = np.zeros(mask.shape, dtype=bool)
    n_comp = 0
    xs, ys, zs = np.where(mask)
    for x, y, z in zip(xs, ys, zs):
        if visited[x, y, z]:
            continue
        n_comp += 1
        stack = [(int(x), int(y), int(z))]
        visited[x, y, z] = True
        while stack:
            cx, cy, cz = stack.pop()
            for dx, dy, dz in ((1,0,0),(-1,0,0),(0,1,0),(0,-1,0),(0,0,1),(0,0,-1)):
                nx, ny, nz = cx + dx, cy + dy, cz + dz
                if 0 <= nx < mask.shape[0] and 0 <= ny < mask.shape[1] and 0 <= nz < mask.shape[2]:
                    if mask[nx, ny, nz] and not visited[nx, ny, nz]:
                        visited[nx, ny, nz] = True
                        stack.append((nx, ny, nz))
    return n_comp

def voxel_metrics(pred_cls, gt_cls):
    pred_cls = pred_cls.astype(np.int16)
    gt_cls = gt_cls.astype(np.int16)
    total_acc = float((pred_cls == gt_cls).mean())
    occupied_mask = gt_cls > 0
    occupied_acc = float((pred_cls[occupied_mask] == gt_cls[occupied_mask]).mean()) if occupied_mask.any() else 0.0
    ious, class_ids = [], []
    for cid in range(1, NUM_CHANNELS):
        p, g = pred_cls == cid, gt_cls == cid
        if not g.any():
            continue
        inter = np.logical_and(p, g).sum()
        union = np.logical_or(p, g).sum()
        ious.append(inter / max(union, 1))
        class_ids.append(cid)
    return {
        'total_acc': total_acc, 'occupied_acc': occupied_acc,
        'miou': float(np.mean(ious)) if ious else 0.0,
        'class_ious': {cid: float(iou) for cid, iou in zip(class_ids, ious)},
    }

def geometry_metrics(pred_cls, gt_cls):
    pred_frag, gt_frag = [], []
    for cid in range(1, NUM_CHANNELS):
        pg, gg = pred_cls == cid, gt_cls == cid
        if gg.any():
            gt_frag.append(count_components(gg))
            pred_frag.append(count_components(pg) if pg.any() else 0)
    return {
        'pred_components_mean': float(np.mean(pred_frag)) if pred_frag else 0.0,
        'gt_components_mean': float(np.mean(gt_frag)) if gt_frag else 0.0,
        'fragmentation_ratio': float(np.mean(pred_frag) / max(np.mean(gt_frag), 1)) if pred_frag else 0.0,
    }

def function_ratio_vector(cls_map):
    occupied = (cls_map > 0).sum()
    if occupied == 0:
        return {}
    return {cid: float((cls_map == cid).sum() / occupied) for cid in range(1, NUM_CHANNELS)}

def function_compliance(pred_cls, gt_cls):
    pred_r, gt_r = function_ratio_vector(pred_cls), function_ratio_vector(gt_cls)
    diffs = [abs(pred_r.get(cid, 0.0) - gt_r.get(cid, 0.0)) for cid in gt_r.keys()]
    return {'ratio_l1': float(np.sum(diffs)), 'ratio_mae': float(np.mean(diffs)) if diffs else 0.0}

@torch.no_grad()
def predict_sample(sample):
    hg = sample['graph']
    hg.y = sample['voxel']
    cond = sample['condition']
    hg.condition = cond.unsqueeze(0) if cond.dim() == 1 else cond
    batch = prepare_graph_batch(hg).to(device)
    logits, _, _ = model(batch)
    pred = logits_to_class(logits)[0]
    gt = torch.argmax(sample['voxel'], dim=0).numpy()
    return pred, gt

def evaluate_one(sample, name=''):
    pred, gt = predict_sample(sample)
    m = {'sample': name}
    m.update(voxel_metrics(pred, gt))
    m.update(geometry_metrics(pred, gt))
    m.update(function_compliance(pred, gt))
    return m, pred, gt

def plot_slice_compare(pred, gt, title, z_idx=None):
    if z_idx is None:
        occupied_z = np.where(np.any(gt > 0, axis=(0, 1)))[0]
        z_idx = int(occupied_z[len(occupied_z) // 2]) if len(occupied_z) else gt.shape[2] // 2
    gt_slice, pred_slice = gt[:, :, z_idx], pred[:, :, z_idx]
    fig, axes = plt.subplots(1, 3, figsize=(14, 4))
    axes[0].imshow(gt_slice.T, origin='lower', cmap='tab20', vmin=0, vmax=NUM_CHANNELS - 1)
    axes[0].set_title(f'GT slice z={z_idx}')
    axes[1].imshow(pred_slice.T, origin='lower', cmap='tab20', vmin=0, vmax=NUM_CHANNELS - 1)
    axes[1].set_title(f'Pred slice z={z_idx}')
    err = (pred_slice != gt_slice).astype(float)
    err[gt_slice == 0] *= 0.3
    axes[2].imshow(err.T, origin='lower', cmap='Reds', vmin=0, vmax=1)
    axes[2].set_title('Error map')
    for ax in axes:
        ax.set_xlabel('X index'); ax.set_ylabel('Y index')
    fig.suptitle(title, fontsize=13, fontweight='bold')
    plt.tight_layout(); plt.show()

print('Step 6a 就绪: evaluate_one 已定义')


In [ ]:
# Step 6b: 运行评估（需先运行 Step 6a、Step 3、Step 4）
if 'evaluate_one' not in globals():
    raise RuntimeError('请先运行 Step 6a（函数定义 cell）')

weight_path = os.path.join(DATA_DIR, 'spatial_modal_cvae_mvp.pth')
model.load_state_dict(torch.load(weight_path, map_location=device, weights_only=True))
model.eval()

if 'val_set' in globals() and val_set:
    eval_set = val_set
elif 'dataset_list' in globals() and dataset_list:
    eval_set = dataset_list[-max(1, len(dataset_list) // 5):]
else:
    raise RuntimeError('请先运行 Step 4 训练')

print(f'评估样本数: {len(eval_set)}')
rows, cache = [], {}
for i, hg in enumerate(eval_set):
    sample = {
        'graph': hg, 'voxel': hg.y,
        'condition': hg.condition.squeeze(0) if hg.condition.dim() > 1 else hg.condition,
    }
    name = f'sample_{i:03d}'
    metrics, pred, gt = evaluate_one(sample, name)
    cache[name] = (metrics, pred, gt)
    rows.append({'sample': name, 'total_acc': metrics['total_acc'], 'occupied_acc': metrics['occupied_acc'],
                 'miou': metrics['miou'], 'ratio_mae': metrics['ratio_mae'],
                 'fragmentation_ratio': metrics['fragmentation_ratio']})

df = pd.DataFrame(rows).sort_values('miou', ascending=False)
summary = df[['total_acc', 'occupied_acc', 'miou', 'ratio_mae', 'fragmentation_ratio']].agg(['mean', 'std'])
print('\n=== 验证集汇总 (mean ± std) ===')
for col in summary.columns:
    print(f"{col:22s}: {summary.loc['mean', col]*100:6.2f} ± {summary.loc['std', col]*100:5.2f} %")

csv_path = os.path.join(EVAL_DIR, 'val_metrics.csv')
df.to_csv(csv_path, index=False, encoding='utf-8-sig')
print(f'\n明细已保存: {csv_path}')
try:
    display(df.head(10))
except NameError:
    print(df.head(10))

all_class_ious = defaultdict(list)
for metrics, _, _ in cache.values():
    for cid, iou in metrics['class_ious'].items():
        all_class_ious[cid].append(iou)
class_rows = []
for cid in sorted(all_class_ious.keys()):
    tname = TYPE_NAMES[cid]
    vals = all_class_ious[cid]
    class_rows.append({'class_id': cid, 'type': tname, 'name_cn': CN_NAMES.get(tname, tname),
                       'iou_mean': np.mean(vals), 'iou_std': np.std(vals)})
df_cls = pd.DataFrame(class_rows).sort_values('iou_mean', ascending=True)
fig, ax = plt.subplots(figsize=(10, 5))
ax.barh(df_cls['name_cn'], df_cls['iou_mean'], xerr=df_cls['iou_std'], color='#4C78A8', alpha=0.85)
ax.set_xlim(0, 1); ax.set_xlabel('IoU'); ax.set_title('各类语义体素 IoU（验证集均值）')
ax.grid(axis='x', alpha=0.3); plt.tight_layout(); plt.show()

best_name, worst_name = df.iloc[0]['sample'], df.iloc[-1]['sample']
for label, sname in [('最佳 mIoU', best_name), ('最差 mIoU', worst_name)]:
    metrics, pred, gt = cache[sname]
    print(f"\n--- {label}: {sname} | mIoU={metrics['miou']*100:.2f}% | occupied={metrics['occupied_acc']*100:.2f}% ---")
    plot_slice_compare(pred, gt, f'{label}: {sname}')

conf = np.zeros((NUM_CHANNELS, NUM_CHANNELS), dtype=np.int64)
for _, pred, gt in cache.values():
    mask = gt > 0
    for p, g in zip(pred[mask], gt[mask]):
        conf[g, p] += 1
conf_norm = conf[1:, 1:].astype(float)
conf_norm = np.divide(conf_norm, np.maximum(conf_norm.sum(axis=1, keepdims=True), 1))
labels = [CN_NAMES.get(TYPE_NAMES[i], TYPE_NAMES[i]) for i in range(1, NUM_CHANNELS)]
fig, ax = plt.subplots(figsize=(9, 7))
im = ax.imshow(conf_norm, cmap='Blues', vmin=0, vmax=1)
ax.set_xticks(range(len(labels))); ax.set_yticks(range(len(labels)))
ax.set_xticklabels(labels, rotation=45, ha='right'); ax.set_yticklabels(labels)
ax.set_xlabel('Predicted'); ax.set_ylabel('Ground Truth')
ax.set_title('非空体素混淆矩阵（行归一化）')
plt.colorbar(im, ax=ax, fraction=0.046); plt.tight_layout(); plt.show()

report = {
    'summary_mean': {k: float(summary.loc['mean', k]) for k in summary.columns},
    'summary_std': {k: float(summary.loc['std', k]) for k in summary.columns},
    'best_sample': best_name, 'worst_sample': worst_name,
    'class_iou_mean': {int(r.class_id): float(r.iou_mean) for r in df_cls.itertuples()},
}
json_path = os.path.join(EVAL_DIR, 'val_summary.json')
with open(json_path, 'w', encoding='utf-8') as f:
    json.dump(report, f, indent=2, ensure_ascii=False)
print(f'摘要已保存: {json_path}')
